# 06. Business Simulation & Replenishment Optimization
## Supply Chain Optimization — FMCG / Retail

---

### Objective
Translate machine learning risk probabilities into operational supply chain actions:
1. Generate warehouse-level **Replenishment Recommendations** with action plans, target quantities, and priority badges.
2. Execute **Supply Reallocation Simulation** pairing surplus and deficit facilities to eliminate stockouts and reduce excess holding capital.
3. Compute simulated ROI, holding cost savings, and shortage reduction percentages.


In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from src.recommendations import ReplenishmentRecommender
from src.simulation import SupplyReallocationSimulator

df_engineered = pd.read_csv("../data/processed/fmcg_supply_chain_engineered.csv")
prod_pipeline = joblib.load("../models/random_forest_pipeline.joblib")

latest_month = df_engineered["date"].max()
df_latest = df_engineered[df_engineered["date"] == latest_month].copy().reset_index(drop=True)
print(f"Evaluating Snapshot for: {latest_month} ({len(df_latest)} Warehouses)")


### 1. Generate Replenishment Recommendations


In [ ]:
recommender = ReplenishmentRecommender(shortage_prob_threshold=0.50)
df_recs = recommender.generate_recommendations(df_latest, prod_pipeline)

# Priority breakdown
display(df_recs["priority"].value_counts())


In [ ]:
# Critical action facilities
critical_facilities = df_recs[df_recs["priority"] == "Critical"]
display(critical_facilities[["warehouse_id", "warehouse_name", "region", "priority", "recommended_action", "recommended_quantity", "explainable_reason"]])


### 2. Lateral Inventory Reallocation Simulation


In [ ]:
simulator = SupplyReallocationSimulator(
    holding_cost_per_unit_month=2.50,
    stockout_penalty_per_unit=8.00,
    intra_region_transfer_cost_per_unit=0.45,
    inter_region_transfer_cost_per_unit=1.10
)
sim_results = simulator.run_reallocation_simulation(df_latest, predicted_risk_col="predicted_risk")

print(f"Shortage Units Reduction: {sim_results['shortage_reduction_pct']}%")
print(f"Overstock Units Reduction: {sim_results['overstock_reduction_pct']}%")
print(f"Total Units Reallocated: {sim_results['total_units_reallocated']:,}")
print(f"Holding Cost Savings: ${sim_results['holding_cost_savings']:,.2f}")
print(f"Net Financial Benefit: ${sim_results['net_financial_benefit']:,.2f}")
print(f"Facilities Participating: {sim_results['affected_warehouses_count']}")


### 3. Lateral Transfer Manifest Sample


In [ ]:
display(sim_results["transfer_log"].head(10))


### 4. Before vs. After Impact Visualization


In [ ]:
categories = ["Shortage Before", "Shortage After", "Overstock Before", "Overstock After"]
values = [
    sim_results["total_shortage_before"] / 1e3,
    sim_results["total_shortage_after"] / 1e3,
    sim_results["total_overstock_before"] / 1e3,
    sim_results["total_overstock_after"] / 1e3
]

plt.figure(figsize=(8, 4.5))
bars = plt.bar(categories, values, color=["#d62728", "#ff9896", "#ff7f0e", "#ffbb78"])
for bar in bars:
    y = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, y + 0.3, f"{y:.1f}k", ha="center", fontweight="bold")
plt.title("Lateral Inventory Reallocation Simulation Impact", fontsize=12, fontweight="bold")
plt.ylabel("Inventory Units (Thousands)")
plt.tight_layout()
plt.show()
